# Inspeção Inicial dos Dados - Hapvida

O objetivo deste notebook é validar a estrutura do dataset e identificar quais colunas precisam de tratamento para os requisitos do projeto.

In [8]:
import sys
import os
sys.path.append(os.path.abspath('..'))

## Observação geral

- A coluna `TEMPO` está como `object` (texto). Precisamos converter para data.
- A coluna `LOCAL` mistura cidade e estado (Ex: `Recife - PE`).
- A coluna `TEMA` não possui padronização de texto, alternando entre valores em maiúsculo e minúsculo.
- A coluna `CATEGORIA` apresenta uma estrutura hierárquica separada pelo delimitador <-> e contém ruídos como aspas duplas.

In [ ]:
from src.entrada_saida import carregar_dados_brutos
import pandas as pd

df = carregar_dados_brutos()

display(df.head())
print(df.info())
print(df.CATEGORIA)

,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
0,149490335,TEMPO DE ATENDIMENTO,Recife - PE,2022-01-09,Demora na execução<->Plano<->Planos de Saúde<-...,Não respondida,Acabei de sair de uma urgência por causa de at...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
1,149499817,Hapvida não tem nutrólogo,Salvador - BA,2022-01-09,Planos de saúde<->Qualidade do serviço prestad...,Não respondida,O Hapvida diz que fornece o serviço de nutrólo...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
2,149498293,Descaso de tratamento de Hemodiálise,Olinda - PE,2022-01-09,"Demora para autorização de consultas, exames e...",Respondida,"Meu irmão Wagner Santiago, estava internado de...",https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
3,149495455,DESORGANIZAÇÃO E FALTA DE RESOLUÇÃO DE PROBLEMA,Goiânia - GO,2022-01-09,Demora na execução<->Planos de saúde<->Planos ...,Não respondida,Agendei pelo chat um procedimento onde fui bem...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
4,149495285,Liberação de Procedimento,Fortaleza - CE,2022-01-09,Planos de saúde<->Planos de Saúde<->Hapvida Sa...,Respondida,Paguei fatura do plano em atraso e tal atraso ...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1016 entries, 0 to 1015
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID             1016 non-null   int64 
 1   TEMA           1016 non-null   object
 2   LOCAL          1016 non-null   object
 3   TEMPO          1016 non-null   object
 4   CATEGORIA      1016 non-null   object
 5   STATUS         1016 non-null   object
 6   DESCRICAO      1016 non-null   object
 7   URL            1016 non-null   object
 8   ANO            1016 non-null   int64 
 9   MES            1016 non-null   int64 
 10  DIA            1016 non-null   int64 
 11  DIA_DO_ANO     1016 non-null   int64 
 12  SEMANA_DO_ANO  1016 non-null   int64 
 13  DIA_DA_SEMANA  1016 non-null   int64 
 14  TRIMETRES      1016 non-null   int64 
 15  CASOS          1016 non-null   int64 
dtypes: int64(9), object(7)
memory usage: 127.1+ KB
None
0       Demora na execução<->Plano<->Planos de Saúde<-...
1   

np.int64(0)

## Análise Temporal

- As colunas `ANO`, `MES`, `DIA`, `DIA_DO_ANO` são redundantes, pois todas as informações já estão contidas em `TEMPO`.
- Validamos que os valores dessas colunas são consistentes com `TEMPO`, portanto podem ser removidas com segurança para manter o dataset mais enxuto.
- Também recomendamos renomear `TEMPO` para `DATA`, nome mais intuitivo e padronizado para uma coluna de data.

In [34]:
df['TEMPO'] = pd.to_datetime(df['TEMPO'])

df['ANO_check']  = df['TEMPO'].dt.year   == df['ANO']
df['MES_check']  = df['TEMPO'].dt.month  == df['MES']
df['DIA_check']  = df['TEMPO'].dt.day    == df['DIA']

df[['ANO_check','MES_check','DIA_check']].value_counts()

print(df.DIA_DO_ANO)

0         9
1         9
2         9
3         9
4         9
       ... 
1011    343
1012    343
1013    343
1014    343
1015    343
Name: DIA_DO_ANO, Length: 1016, dtype: int64


## Análise da Coluna `STATUS`

- A coluna `STATUS` é categórica e possui 5 valores únicos: `Não respondida`, `Respondida`, `Resolvido`, `Em réplica` e `Não resolvido`.
- Para facilitar análises e KPIs, criaremos uma coluna binária `RESOLVIDO` onde `True` representa reclamações resolvidas e `False` representa pendentes — evitando repetir essa lógica em todo o projeto.

In [36]:
df['STATUS'].value_counts()

STATUS
Não respondida    639
Respondida        271
Resolvido          67
Em réplica         36
Não resolvido       3
Name: count, dtype: int64

## Análise da Coluna `CASOS`

- A coluna `CASOS` é numérica e seus valores variam entre 7 e 75 — ou seja, não é unitária (cada linha não representa 1 reclamação).
- Pode representar o volume total de reclamações agrupadas por dia/tema.
- Se não for tratada corretamente, somas e médias podem gerar distorções nos gráficos.
- Precisamos entender o que cada linha realmente representa antes de usar essa coluna.

In [44]:
print('Valores:',sorted(df['CASOS'].unique()))
print('Valores únicos:', df['CASOS'].nunique())
print('Mínimo:', df['CASOS'].min())
print('Máximo:', df['CASOS'].max())

Valores: [np.int64(7), np.int64(13), np.int64(14), np.int64(15), np.int64(18), np.int64(24), np.int64(26), np.int64(49), np.int64(52), np.int64(59), np.int64(60), np.int64(61), np.int64(64), np.int64(65), np.int64(66), np.int64(68), np.int64(71), np.int64(75)]
Valores únicos: 18
Mínimo: 7
Máximo: 75
